In [0]:
'''
Passo: configurar ambiente, parâmetros e volumes desta live.
O que observar: esta parte do projeto prepara dois volumes com papéis diferentes.
Validar: um volume guarda os lotes gerados e outro simula a chegada de arquivos para o bronze observar.
Sinal de erro: misturar staging e inbox no mesmo path e perder a narrativa de ingestão.
'''

from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timedelta
import random

CATALOG = 'aulas_ao_vivo'
SCHEMA = 'live_20260413'
VOLUME_STAGING = 'armazenamento_pedidos_staging'
VOLUME_INBOX = 'armazenamento_pedidos_inbox'

STAGING_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_STAGING}'
INBOX_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_INBOX}'

JSON_STAGING_PATH = f'{STAGING_PATH}/pedidos/json'
PARQUET_STAGING_PATH = f'{STAGING_PATH}/pedidos/parquet'
JSON_INBOX_PATH = f'{INBOX_PATH}/pedidos/json'
PARQUET_INBOX_PATH = f'{INBOX_PATH}/pedidos/parquet'

TOTAL_LOTES = 5
PEDIDOS_UNICOS_POR_LOTE = 25
TAXA_DUPLICIDADE = 0.24
TAXA_NULOS = 0.12
SEED = 42

'''
ASSUNÇÃO:
- O catálogo `aulas_ao_vivo` já existe no ambiente.
- O bronze futuro observará os paths do inbox, e não os paths de staging.
- Nesta parte da live, a chegada de novos arquivos será emulada movendo lotes do staging para o inbox.
'''

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME_STAGING}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME_INBOX}`')

print(f'Staging volume: {STAGING_PATH}')
print(f'Inbox volume:   {INBOX_PATH}')
print(f'JSON staging:   {JSON_STAGING_PATH}')
print(f'Parquet staging:{PARQUET_STAGING_PATH}')
print(f'JSON inbox:     {JSON_INBOX_PATH}')
print(f'Parquet inbox:  {PARQUET_INBOX_PATH}')


In [0]:
'''
Passo: recriar a estrutura de diretórios usada na demo.
O que observar: o staging começa populado e o inbox começa vazio para o bronze perceber a chegada incremental.
Validar: os quatro diretórios principais precisam existir após a execução.
Sinal de erro: reaproveitar lixo de execuções anteriores e confundir o comportamento do Auto Loader.
'''

def reset_path(path: str):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass
    dbutils.fs.mkdirs(path)

for path in [
    JSON_STAGING_PATH,
    PARQUET_STAGING_PATH,
    JSON_INBOX_PATH,
    PARQUET_INBOX_PATH
]:
    reset_path(path)

print('Diretórios recriados com sucesso.')
print('O staging está pronto para receber os lotes gerados.')
print('O inbox ficou vazio para simular ingestão incremental.')


In [0]:
'''
Passo: definir a função de geração de pedidos mockados para esta live.
O que observar: a função herda a ideia da aula anterior, mas agora também injeta nulos em colunas importantes.
Validar: `pedido_id` segue como chave observável e as falhas didáticas ficam visíveis na amostra.
Sinal de erro: remover os problemas na origem e perder material para bronze e silver.
'''

def gerar_pedidos_mock_lote(total_pedidos_unicos: int, taxa_duplicidade: float, taxa_nulos: float, lote_id: int, seed: int = 42):
    if total_pedidos_unicos <= 0:
        raise ValueError('`total_pedidos_unicos` deve ser maior que zero.')

    if not (0 <= taxa_duplicidade <= 1):
        raise ValueError('`taxa_duplicidade` deve estar entre 0 e 1.')

    if not (0 <= taxa_nulos <= 1):
        raise ValueError('`taxa_nulos` deve estar entre 0 e 1.')

    if lote_id <= 0:
        raise ValueError('`lote_id` deve ser maior que zero.')

    rng = random.Random(seed + lote_id)

    produtos = ['notebook', 'monitor', 'teclado', 'mouse', 'headset']
    status_pedido = ['criado', 'pago', 'faturado', 'enviado']
    canais_venda = ['site', 'app', 'marketplace']
    data_base = datetime(2026, 4, 1, 8, 0, 0) + timedelta(days=lote_id - 1)
    primeiro_id = ((lote_id - 1) * total_pedidos_unicos) + 1

    schema = T.StructType([
        T.StructField('pedido_id', T.StringType(), False),
        T.StructField('cliente_id', T.StringType(), True),
        T.StructField('produto', T.StringType(), False),
        T.StructField('valor_pedido', T.DoubleType(), True),
        T.StructField('status_pedido', T.StringType(), True),
        T.StructField('created_at', T.TimestampType(), False),
        T.StructField('updated_at', T.TimestampType(), False),
        T.StructField('canal_venda', T.StringType(), False),
        T.StructField('lote_id', T.IntegerType(), False)
    ])

    registros_base = []

    for i in range(total_pedidos_unicos):
        identificador = primeiro_id + i
        created_at = data_base + timedelta(minutes=i * 7)
        updated_at = created_at + timedelta(minutes=rng.randint(0, 180))

        registros_base.append({
            'pedido_id': f'P{identificador:06d}',
            'cliente_id': f'C{rng.randint(1, max(10, total_pedidos_unicos // 2)):05d}',
            'produto': rng.choice(produtos),
            'valor_pedido': round(rng.uniform(50, 1500), 2),
            'status_pedido': rng.choice(status_pedido),
            'created_at': created_at,
            'updated_at': updated_at,
            'canal_venda': rng.choice(canais_venda),
            'lote_id': lote_id
        })

    total_nulos = int(total_pedidos_unicos * taxa_nulos)
    campos_com_nulos = ['cliente_id', 'valor_pedido', 'status_pedido']

    if total_nulos > 0:
        indices_nulos = rng.sample(range(len(registros_base)), total_nulos)
        for indice in indices_nulos:
            campo_escolhido = rng.choice(campos_com_nulos)
            registros_base[indice][campo_escolhido] = None

    total_duplicados = int(total_pedidos_unicos * taxa_duplicidade)
    registros_duplicados = []

    if total_duplicados > 0:
        for registro in rng.sample(registros_base, total_duplicados):
            registros_duplicados.append(dict(registro))

    registros_finais = list(registros_base) + list(registros_duplicados)
    rng.shuffle(registros_finais)

    return spark.createDataFrame(registros_finais, schema=schema)


In [0]:
'''
Passo: criar helpers para gravar lotes em JSON e Parquet e resumir a qualidade do lote.
O que observar: cada lote será gravado como uma pasta independente para facilitar o movimento entre staging e inbox.
Validar: os arquivos ficam separados por formato e por lote.
Sinal de erro: misturar formatos no mesmo diretório e perder clareza sobre o fluxo de ingestão.
'''

def gravar_lote(df, base_path: str, formato: str, lote_id: int) -> str:
    destino = f'{base_path}/lote_{lote_id:03d}'
    writer = df.coalesce(1).write.mode('overwrite')

    if formato == 'json':
        writer.json(destino)
    elif formato == 'parquet':
        writer.parquet(destino)
    else:
        raise ValueError('Formato inválido. Use `json` ou `parquet`.')

    return destino

def diagnostico_lote(df, lote_id: int):
    total_linhas = df.count()
    pedidos_distintos = df.select('pedido_id').distinct().count()
    linhas_duplicadas = total_linhas - pedidos_distintos

    nulos = df.select(
        F.sum(F.col('cliente_id').isNull().cast('int')).alias('nulos_cliente_id'),
        F.sum(F.col('valor_pedido').isNull().cast('int')).alias('nulos_valor_pedido'),
        F.sum(F.col('status_pedido').isNull().cast('int')).alias('nulos_status_pedido')
    ).collect()[0].asDict()

    return {
        'lote_id': lote_id,
        'total_linhas': total_linhas,
        'pedidos_distintos': pedidos_distintos,
        'linhas_duplicadas': linhas_duplicadas,
        **nulos
    }


In [0]:
'''
Passo: gerar os lotes sintéticos e gravar cada lote nos dois formatos no volume de staging.
O que observar: o mesmo conjunto lógico é salvo em JSON e Parquet para permitir duas leituras na demo.
Validar: devem existir 5 lotes em JSON e 5 lotes em Parquet após a execução.
Sinal de erro: gerar apenas um arquivo grande e perder a simulação de chegada incremental.
'''

resumo_lotes = []

for lote_id in range(1, TOTAL_LOTES + 1):
    df_lote = gerar_pedidos_mock_lote(
        total_pedidos_unicos=PEDIDOS_UNICOS_POR_LOTE,
        taxa_duplicidade=TAXA_DUPLICIDADE,
        taxa_nulos=TAXA_NULOS,
        lote_id=lote_id,
        seed=SEED
    )

    json_path = gravar_lote(df_lote, JSON_STAGING_PATH, 'json', lote_id)
    parquet_path = gravar_lote(df_lote, PARQUET_STAGING_PATH, 'parquet', lote_id)

    resumo = diagnostico_lote(df_lote, lote_id)
    resumo['json_path'] = json_path
    resumo['parquet_path'] = parquet_path
    resumo_lotes.append(resumo)

resumo_lotes_df = spark.createDataFrame(resumo_lotes)
display(resumo_lotes_df.orderBy('lote_id'))

print('Geração concluída.')
print(f'Lotes gerados em JSON: {TOTAL_LOTES}')
print(f'Lotes gerados em Parquet: {TOTAL_LOTES}')


In [0]:
'''
Passo: validar o conteúdo de um lote e confirmar que as falhas didáticas estão presentes.
O que observar: a amostra precisa mostrar a mesma base da live anterior, agora organizada para ingestão por camadas.
Validar: schema, amostra, nulos e duplicidade aparecem com evidência objetiva.
Sinal de erro: seguir para o bronze sem provar que a base de origem está fiel ao cenário da aula.
'''

lote_json_01_df = spark.read.json(f'{JSON_STAGING_PATH}/lote_001')
lote_parquet_01_df = spark.read.parquet(f'{PARQUET_STAGING_PATH}/lote_001')

print('Schema do lote JSON:')
lote_json_01_df.printSchema()

display(lote_json_01_df.orderBy('pedido_id').limit(10))
display(lote_parquet_01_df.orderBy('pedido_id').limit(10))

nulos_lote_01_df = lote_json_01_df.select(
    F.sum(F.col('cliente_id').isNull().cast('int')).alias('nulos_cliente_id'),
    F.sum(F.col('valor_pedido').isNull().cast('int')).alias('nulos_valor_pedido'),
    F.sum(F.col('status_pedido').isNull().cast('int')).alias('nulos_status_pedido')
)

duplicidade_lote_01_df = lote_json_01_df.agg(
    F.count('*').alias('total_linhas'),
    F.countDistinct('pedido_id').alias('pedidos_distintos'),
    (F.count('*') - F.countDistinct('pedido_id')).alias('linhas_duplicadas')
)

display(nulos_lote_01_df)
display(duplicidade_lote_01_df)


In [0]:
'''
Passo: criar funções para inspecionar e publicar lotes do staging para o inbox.
O que observar: o bronze futuro deve observar apenas o inbox; por isso mover os lotes é o gatilho da demo.
Validar: a publicação move uma pasta de lote inteira, preservando a estrutura de arquivos criada pelo Spark.
Sinal de erro: ler direto do staging e perder a simulação de chegada incremental.
'''

def listar_lotes(path: str):
    return [item.name.rstrip('/') for item in dbutils.fs.ls(path)]

def publicar_lote(formato: str, lote_id: int):
    formato_normalizado = formato.strip().lower()

    if formato_normalizado not in ['json', 'parquet']:
        raise ValueError('Formato inválido. Use `json` ou `parquet`.')

    origem_base = JSON_STAGING_PATH if formato_normalizado == 'json' else PARQUET_STAGING_PATH
    destino_base = JSON_INBOX_PATH if formato_normalizado == 'json' else PARQUET_INBOX_PATH

    origem = f'{origem_base}/lote_{lote_id:03d}'
    destino = f'{destino_base}/lote_{lote_id:03d}'

    dbutils.fs.mv(origem, destino, True)
    print(f'Lote publicado: {origem} -> {destino}')

def publicar_todos_os_lotes(formato: str):
    for nome_lote in sorted(listar_lotes(JSON_STAGING_PATH if formato == 'json' else PARQUET_STAGING_PATH)):
        lote_id = int(nome_lote.replace('lote_', ''))
        publicar_lote(formato, lote_id)

print('Lotes no JSON staging:   ', listar_lotes(JSON_STAGING_PATH))
print('Lotes no Parquet staging:', listar_lotes(PARQUET_STAGING_PATH))
print('Lotes no JSON inbox:     ', listar_lotes(JSON_INBOX_PATH))
print('Lotes no Parquet inbox:  ', listar_lotes(PARQUET_INBOX_PATH))
